In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import sqlalchemy as sa
import os
from natsort import natsorted
from natsort import natsort_keygen
from fern_raw_db_dompare import *

In [3]:
# --- Main workflow ---
# --- read data ---
meta_path = '/workspaces/crmprtd/fern/FERNNorth2025_insert/tables/20241125 MeteorologicalNetworks-FERN-VF-shared.xlsx'

df_station = pd.read_excel(meta_path)
df_station = df_station.sort_values(by='station_name', key=natsort_keygen())


# --- output folder ---
output_folder = '/workspaces/crmprtd/fern/FERNNorth2025_insert/output/'
os.makedirs(output_folder, exist_ok=True)


### 1. First match the varible of each station with itself (same station, same variable, compare the start and end time)

In [4]:
raw_summary_path = os.path.join(output_folder, "Fern_raw_station_variable_summary.csv")
db_summary_path = os.path.join(output_folder, "Fern_db_station_variable_summary.csv")

raw_summary = pd.read_csv(raw_summary_path)
db_summary = pd.read_csv(db_summary_path)

all_matches = []

for station_name in df_station["station_name"]:
    raw_one = raw_summary.loc[raw_summary["station_name"] == station_name]
    db_one = db_summary.loc[db_summary["station_name"] == station_name]
    print(db_one)

    station_matches = fuzzy_match_station_vars(raw_one, db_one, threshold=80)
    all_matches.append(station_matches)

# Combine all station matches
# Filter out empty DataFrames before concatenation
non_empty_matches = [df for df in all_matches if not df.empty]

if non_empty_matches:
    matches_df = pd.concat(non_empty_matches, ignore_index=True)
else:
    matches_df = pd.DataFrame(columns=["station_name", "variable", "db_var_match", "score"])
    
# Merge with raw_summary
raw_summary_matched = raw_summary.merge(matches_df, on=["station_name", "variable"], how="left")


Empty DataFrame
Columns: [station_name, native_id, lat, lon, elev, net_var_name, unit, earliest_time, latest_time]
Index: []
    station_name native_id        lat         lon    elev  \
0       BarrenWx    Barren  54.510444 -126.614341  1051.0   
28      BarrenWx    Barren  54.510444 -126.614341  1051.0   
56      BarrenWx    Barren  54.510444 -126.614341  1051.0   
84      BarrenWx    Barren  54.510444 -126.614341  1051.0   
112     BarrenWx    Barren  54.510444 -126.614341  1051.0   
144     BarrenWx    Barren  54.510444 -126.614341  1051.0   
149     BarrenWx    Barren  54.510444 -126.614341  1051.0   
177     BarrenWx    Barren  54.510444 -126.614341  1051.0   
207     BarrenWx    Barren  54.510444 -126.614341  1051.0   
225     BarrenWx    Barren  54.510444 -126.614341  1051.0   
228     BarrenWx    Barren  54.510444 -126.614341  1051.0   
242     BarrenWx    Barren  54.510444 -126.614341  1051.0   
270     BarrenWx    Barren  54.510444 -126.614341  1051.0   

                  ne

In [5]:
non_empty_matches

[   station_name         variable      db_var_match       score
 0      BarrenWx             Rain            Rainmm   80.000000
 1      BarrenWx         Pressure      Pressurembar   80.000000
 2      BarrenWx             Temp             TempC   88.888889
 3      BarrenWx               RH                RH  100.000000
 4      BarrenWx            DewPt            DewPtC   90.909091
 5      BarrenWx       Wind Speed       WindSpeedms   85.714286
 6      BarrenWx       Gust Speed       GustSpeedms   85.714286
 7      BarrenWx   Wind Direction     WindDirection   96.296296
 8      BarrenWx  Solar Radiation  SolarRadiationWm   90.322581
 9      BarrenWx    Water Content              None   64.864865
 10     BarrenWx           WC_cal              None   26.666667
 11     BarrenWx        Soil Temp         Soil_Temp   88.888889
 12     BarrenWx         Sfc Temp              None   61.538462
 13     BarrenWx    Water Content              None   64.864865
 14     BarrenWx    Water Content       

In [6]:
#### merging the information in the database into the same match form
raw_summary_enriched = raw_summary_matched.merge(
    db_summary[['station_name','native_id', 'net_var_name', 'unit', 'earliest_time', 'latest_time']],
    left_on=['station_name', 'db_var_match'],   # match station + variable
    right_on=['station_name', 'net_var_name'],
    how='left'
)
raw_summary_enriched = raw_summary_enriched.drop(columns=['net_var_name'])

raw_summary_enriched = raw_summary_enriched.rename(
    columns={
        "unit_x": "unit_raw",
        'native_id_x':'native_id_raw',
        'native_id_y': 'native_id_db',
        "earliest_time_x": "earliest_time_raw",
        "latest_time_x": "latest_time_raw",
        "unit_y": "unit_db",
        "earliest_time_y": "earliest_time_db",
        "latest_time_y": "latest_time_db",
        "score": "variable_match_score",

    }
)

# Convert to datetime if they’re strings
for col in ["earliest_time_raw", "latest_time_raw", "earliest_time_db", "latest_time_db"]:
    raw_summary_enriched[col] = pd.to_datetime(raw_summary_enriched[col], errors="coerce")

# Calculate time gaps (in days)
raw_summary_enriched["earliest_diff_days"] = (
    (raw_summary_enriched["earliest_time_raw"] - raw_summary_enriched["earliest_time_db"]).dt.days
)

raw_summary_enriched["latest_diff_days"] = (
    (raw_summary_enriched["latest_time_raw"] - raw_summary_enriched["latest_time_db"]).dt.days
)
raw_summary_enriched = raw_summary_enriched.drop_duplicates()

raw_summary_enriched["batch"] = raw_summary_enriched["db_var_match"].apply(
    lambda x: " " if pd.isna(x) else "batch1"
)

raw_summary_enriched = raw_summary_enriched.drop(columns=['native_id_db'])

# Save
# output_path = os.path.join(output_folder, "Fern_raw_db_station_matched_v1.csv")
# raw_summary_enriched.to_csv(output_path, index=False)

# print(f"✅ Done. Matches saved to {output_path}")

In [7]:
batch1 = raw_summary_enriched.loc[raw_summary_enriched['batch'] == 'batch1']
batch1 = batch1.drop(columns=['variable_match_score'])
batch1

,station_name,native_id_raw,station_file_name,lat,lon,elev,variable,unit_raw,earliest_time_raw,latest_time_raw,unit_origin,db_var_match,unit_db,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days,batch
13,BarrenWx,Barren,BarrenWx,54.510444,-126.614341,1051,Rain,mm,2021-07-09 13:00:00,2025-09-24 12:00:00,mm,Rainmm,mm,2021-07-09 13:00:00,2023-09-18 12:00:00,0.0,737.0,batch1
14,BarrenWx,Barren,BarrenWx,54.510444,-126.614341,1051,Pressure,mbar,2021-07-09 12:00:00,2025-09-24 12:00:00,mbar,Pressurembar,millibar,2021-07-09 12:00:00,2023-09-18 12:00:00,0.0,737.0,batch1
15,BarrenWx,Barren,BarrenWx,54.510444,-126.614341,1051,Temp,°C,2021-07-09 12:00:00,2025-09-24 12:00:00,°C,TempC,celsius,2021-07-09 12:00:00,2023-09-18 12:00:00,0.0,737.0,batch1
16,BarrenWx,Barren,BarrenWx,54.510444,-126.614341,1051,RH,%,2021-07-09 12:00:00,2025-09-24 12:00:00,%,RH,%,2021-07-09 12:00:00,2023-09-18 12:00:00,0.0,737.0,batch1
17,BarrenWx,Barren,BarrenWx,54.510444,-126.614341,1051,DewPt,°C,2021-07-09 12:00:00,2025-09-24 12:00:00,°C,DewPtC,celsius,2021-07-09 12:00:00,2023-09-18 12:00:00,0.0,737.0,batch1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
549,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Wind Speed,m/s,2007-07-31 15:00:00,2025-10-30 10:00:00,m/s,WindSpeedms,m s-1,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1
550,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Gust Speed,m/s,2007-07-31 15:00:00,2025-10-30 10:00:00,m/s,GustSpeedms,m s-1,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1
551,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Wind Direction,ø,2007-07-31 15:00:00,2025-10-30 10:00:00,ø,WindDirection,degree,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1
552,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Solar Radiation,W/m²,2007-07-31 15:00:00,2025-10-30 10:00:00,W/m²,SolarRadiationWm,W m-2,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1


In [8]:
count = (batch1['batch'] == 'batch1').sum()
count

np.int64(271)

### 2. Mapping the variables that not in this station, but in all stations

In [9]:
mapping_df = (
    raw_summary_enriched[['variable', 'db_var_match']]
    .dropna(subset=['db_var_match'])      # remove NaN mappings
    .drop_duplicates(subset=['variable']) # remove duplicates by variable
    .reset_index(drop=True)
)

mapping_df

,variable,db_var_match
0,Rain,Rainmm
1,Pressure,Pressurembar
2,Temp,TempC
3,RH,RH
4,DewPt,DewPtC
5,Wind Speed,WindSpeedms
6,Gust Speed,GustSpeedms
7,Wind Direction,WindDirection
8,Solar Radiation,SolarRadiationWm
9,Soil Temp,Soil_Temp


In [10]:
batch2 = raw_summary_enriched[raw_summary_enriched['db_var_match'].isna()]

var_db = raw_summary_enriched["db_var_match"].dropna().drop_duplicates()

# === 4. Prepare variables for fuzzy matching ===
threshold = 80
results = []

# === 5. Perform fuzzy matching ===
for file_var in batch2["variable"]:
    match_result = process.extractOne(file_var, var_db, scorer=fuzz.ratio)
    if match_result:
        match, score, _ = match_result
    else:
        match, score = None, 0

    results.append({
        "variable": file_var,
        "db_var_match": match if score >= threshold else None,
    })

# === 6. Merge results back into batch2 ===
results_df = pd.DataFrame(results)

batch2 = (
    batch2
    .drop(columns=["db_var_match", "score"], errors="ignore")
    .merge(results_df, on="variable", how="left")
    .drop_duplicates()
)

# === 7. Fill in unit_db using match_1_pd lookup ===
unit_lookup = (
    raw_summary_enriched[["db_var_match", "unit_db"]]
    .dropna(subset=["db_var_match"])
    .drop_duplicates()
)

batch2 = (
    batch2
    .drop(columns=["unit_db"], errors="ignore")
    .merge(unit_lookup, on="db_var_match", how="left")
)

# === 8. Reorder columns ===
cols = list(batch2.columns)
# Move db_var_match right after latest_time_raw
if "db_var_match" in cols and "latest_time_raw" in cols:
    cols.insert(cols.index("latest_time_raw") + 1, cols.pop(cols.index("db_var_match")))
# Move unit_db right after db_var_match
if "db_var_match" in cols and "unit_db" in cols:
    cols.insert(cols.index("db_var_match") + 1, cols.pop(cols.index("unit_db")))
batch2 = batch2[cols]

batch2["batch"] = batch2["db_var_match"].apply(
    lambda x: " " if pd.isna(x) else "batch2"
)

batch2 = batch2.drop(columns=['variable_match_score'])

In [11]:
count = (batch2['batch'] == 'batch2').sum()
count

np.int64(92)

### 3. Batch 3, mapping Water_Content_m3_m3, 5, 15, 30cm

In [12]:
batch3 = batch2[batch2['db_var_match'].isna()]

In [13]:
engine = sa.create_engine("postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)


query = sa.text("""
    SELECT net_var_name, standard_name, short_name, unit
    FROM meta_vars
    WHERE network_id = 11;
""")

with engine.connect() as conn:
    var_db = pd.read_sql(query, conn)
    
var_db

,net_var_name,standard_name,short_name,unit
0,RHx,relative_humidity,relative_humidity_maximum,%
1,WindSpeedms,wind_speed,wind_speed_mean,m s-1
2,Wind_n,wind_speed,wind_speed_minimum,m s-1
3,RH,relative_humidity,relative_humidity_mean,%
4,DewPtC,dew_point_temperature,dew_point_temperature_mean,celsius
5,Pressurembar,air_pressure,air_pressure_point,millibar
6,Rainmm,lwe_thickness_of_precipitation_amount,lwe_thickness_of_precipitation_amount_sum,mm
7,Tm,air_temperature,air_temperature_mean,celsius
8,Tx,air_temperature,air_temperature_maximum,celsius
9,Tn,air_temperature,air_temperature_minimum,celsius


In [14]:
import re

# Define mapping for depth -> db_var_match
depth_map = {
    "5 cm": "Water_Content_m3_m3_5_cm",
    "15 cm": "Water_Content_m3_m3_15_cm",
    "30 cm": "Water_Content_m3_m3_30_cm",
}

for depth, db_var in depth_map.items():
    mask = (
        batch3["variable"].eq("Water Content")
        & batch3["unit_raw"].str.match(fr"m.?/m.? {depth}", na=False)
    )
    
    batch3.loc[mask, "db_var_match"] = db_var
    
    # get unit from var_db if exists
    match = var_db.loc[var_db["net_var_name"] == db_var, "unit"]
    if not match.empty:
        batch3.loc[mask, "unit_db"] = match.iloc[0]

batch3["batch"] = batch3["db_var_match"].apply(
    lambda x: " " if pd.isna(x) else "batch3"
)

/tmp/ipykernel_70484/1054742911.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  batch3["batch"] = batch3["db_var_match"].apply(


In [15]:
count = (batch3['batch'] == 'batch3').sum()
count

np.int64(33)

### 4.Manually match

In [21]:
batch4 = batch3[batch3['db_var_match'].isna()]

depth_map = {
    # "Wetness": "Wetness",
    "WC_cal": "Water_Content_Cal_m3_m3_15_cm",
    "Rain raw": "Rainmm_raw",
    "Air Temp": "TempC",
    "Sfc Temp": "Surface_Temp",
    "Sfc temp": "Surface_Temp",
    # "Gd Temp 25 cm": "Soil_Temp_25cm",
    "Soil Temp 50 cm": "Soil_Temp_50cm",
    "Soil Temp 75 cm": "Soil_Temp_75cm",
    "Soil temp" : "Soil_Temp",
    'Temp 30 cm': "TempC_30_cm",
    'RH 30 cm': "RH_30cm",
    'DewPt 30 cm': "DewPt_30cm"

}

for variable_name, db_var in depth_map.items():
    mask = (
        batch4["variable"].eq(variable_name)
    )
    
    batch4.loc[mask, "db_var_match"] = db_var
    
    # get unit from var_db if exists
    match = var_db.loc[var_db["net_var_name"] == db_var, "unit"]
    if not match.empty:
        batch4.loc[mask, "unit_db"] = match.iloc[0]

batch4["batch"] = batch4["db_var_match"].apply(
    lambda x: " " if pd.isna(x) else "batch4"
)

batch4 = batch4[batch4['variable'] != 'Snow depth raw'].reset_index(drop=True)
batch4 = batch4[batch4['variable'] != 'Snow sensor Temp'].reset_index(drop=True)


/tmp/ipykernel_70484/202477047.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  batch4["batch"] = batch4["db_var_match"].apply(


In [22]:
batch4
count = (batch4['batch'] == 'batch4').sum()
count

np.int64(80)

### 5. The left is batch5, are the varibles ends with 2, means two sensers

In [23]:
batch4

batch5 = batch4[batch4['db_var_match'].isna()]
batch5["batch"] = "batch5"

batch5

/tmp/ipykernel_70484/3931571217.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  batch5["batch"] = "batch5"


,station_name,native_id_raw,station_file_name,lat,lon,elev,variable,unit_raw,earliest_time_raw,latest_time_raw,db_var_match,unit_db,unit_origin,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days,batch
1,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,Temp 2,°C,2011-07-28 09:00:00,2024-08-10 19:00:00,None,NaN,°C,NaT,NaT,NaN,NaN,batch5
10,BlackhawkWx,Blackhawk,BlackhawkWx,55.078885,-120.352171,962,Temp 2,°C,2024-07-18 10:00:00,2025-07-24 09:00:00,None,NaN,°C,NaT,NaT,NaN,NaN,batch5
11,BlackhawkWx,Blackhawk,BlackhawkWx,55.078885,-120.352171,962,RH 2,%,2024-07-18 10:00:00,2025-07-24 09:00:00,None,NaN,%,NaT,NaT,NaN,NaN,batch5
12,BlackhawkWx,Blackhawk,BlackhawkWx,55.078885,-120.352171,962,DewPt 2,°C,2024-07-18 10:00:00,2025-07-24 09:00:00,None,NaN,°C,NaT,NaT,NaN,NaN,batch5
19,BowronPit,BowronPit,BowronPit,53.902111,-122.015389,722,Temp 2,°C,2024-10-16 14:00:00,2025-09-05 11:00:00,None,NaN,°C,NaT,NaT,NaN,NaN,batch5
20,BowronPit,BowronPit,BowronPit,53.902111,-122.015389,722,RH 2,%,2024-10-16 14:00:00,2025-09-05 11:00:00,None,NaN,%,NaT,NaT,NaN,NaN,batch5
21,BowronPit,BowronPit,BowronPit,53.902111,-122.015389,722,DewPt 2,°C,2024-10-16 14:00:00,2025-09-05 11:00:00,None,NaN,°C,NaT,NaT,NaN,NaN,batch5
26,Canoe Mountain Stn,Canoe Mountain Stn,Canoe Mountain Stn,52.709820,-119.191260,2210,Temp 2,°C,2023-09-14 10:00:00,2025-09-12 08:00:00,None,NaN,°C,NaT,NaT,NaN,NaN,batch5
27,Canoe Mountain Stn,Canoe Mountain Stn,Canoe Mountain Stn,52.709820,-119.191260,2210,RH 2,%,2023-09-14 10:00:00,2025-09-12 08:00:00,None,NaN,%,NaT,NaT,NaN,NaN,batch5
28,Canoe Mountain Stn,Canoe Mountain Stn,Canoe Mountain Stn,52.709820,-119.191260,2210,DewPt 2,°C,2023-09-14 10:00:00,2025-09-12 08:00:00,None,NaN,°C,NaT,NaT,NaN,NaN,batch5


### Combine different batches into 1

In [24]:
combined = pd.concat(
    [
        batch1,
        batch2.loc[batch2['batch'] == 'batch2'],
        batch3.loc[batch3['batch'] == 'batch3'],
        batch4.loc[batch4['batch'] == 'batch4'],
        batch5.loc[batch5['batch'] == 'batch5'],
    ],
    ignore_index=True

)

combined = combined.sort_values(
    by=['station_name', 'variable'],
    ascending=[True, True]
).reset_index(drop=True)


combined.drop(columns=['unit_raw'])

combined = combined.sort_values(by=['station_name', 'variable'])

combined

,station_name,native_id_raw,station_file_name,lat,lon,elev,variable,unit_raw,earliest_time_raw,latest_time_raw,unit_origin,db_var_match,unit_db,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days,batch
0,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,DewPt,°C,2011-07-28 09:00:00,2024-08-10 19:00:00,°C,DewPtC,celsius,NaT,NaT,NaN,NaN,batch2
1,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,Gust Speed,m/s,2010-08-20 19:00:00,2024-08-10 19:00:00,m/s,GustSpeedms,m s-1,NaT,NaT,NaN,NaN,batch2
2,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,RH,%,2011-07-28 09:00:00,2024-08-10 19:00:00,%,RH,%,NaT,NaT,NaN,NaN,batch2
3,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,Rain,mm,2010-08-20 20:00:00,2024-08-10 19:00:00,mm,Rainmm,mm,NaT,NaT,NaN,NaN,batch2
4,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,Sfc Temp,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,°C,Surface_Temp,celsius,NaT,NaT,NaN,NaN,batch4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
528,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Solar Radiation,W/m²,2007-07-31 15:00:00,2025-10-30 10:00:00,W/m²,SolarRadiationWm,W m-2,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1
529,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Temp,°C,2007-07-31 15:00:00,2025-10-30 10:00:00,°C,TempC,celsius,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1
530,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Wetness,%,2008-06-12 13:00:00,2025-10-30 10:00:00,%,Wetness,%,2008-06-12 13:00:00,2023-10-17 09:00:00,0.0,744.0,batch1
531,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Wind Direction,ø,2007-07-31 15:00:00,2025-10-30 10:00:00,ø,WindDirection,degree,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1


In [25]:
db_match_path = os.path.join(output_folder, "Fern_raw_db_station_matched.csv")
combined.to_csv(db_match_path, index=False)
print(f"✅ match saved to {db_match_path}")

combined

✅ match saved to /workspaces/crmprtd/fern/FERNNorth2025_insert/output/Fern_raw_db_station_matched.csv


,station_name,native_id_raw,station_file_name,lat,lon,elev,variable,unit_raw,earliest_time_raw,latest_time_raw,unit_origin,db_var_match,unit_db,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days,batch
0,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,DewPt,°C,2011-07-28 09:00:00,2024-08-10 19:00:00,°C,DewPtC,celsius,NaT,NaT,NaN,NaN,batch2
1,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,Gust Speed,m/s,2010-08-20 19:00:00,2024-08-10 19:00:00,m/s,GustSpeedms,m s-1,NaT,NaT,NaN,NaN,batch2
2,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,RH,%,2011-07-28 09:00:00,2024-08-10 19:00:00,%,RH,%,NaT,NaT,NaN,NaN,batch2
3,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,Rain,mm,2010-08-20 20:00:00,2024-08-10 19:00:00,mm,Rainmm,mm,NaT,NaT,NaN,NaN,batch2
4,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,Sfc Temp,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,°C,Surface_Temp,celsius,NaT,NaT,NaN,NaN,batch4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
528,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Solar Radiation,W/m²,2007-07-31 15:00:00,2025-10-30 10:00:00,W/m²,SolarRadiationWm,W m-2,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1
529,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Temp,°C,2007-07-31 15:00:00,2025-10-30 10:00:00,°C,TempC,celsius,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1
530,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Wetness,%,2008-06-12 13:00:00,2025-10-30 10:00:00,%,Wetness,%,2008-06-12 13:00:00,2023-10-17 09:00:00,0.0,744.0,batch1
531,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Wind Direction,ø,2007-07-31 15:00:00,2025-10-30 10:00:00,ø,WindDirection,degree,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1
